# Exchangeability Experiments

This tests some basic autoregressive models for exchangeability properties.

# Dataset

We create a data set that samples from two probability distributions.  One is a Bernoulli that has p=0.6 for 1 and the other is a Bernoulli that has p=0.4 for 1.  The mixture is 75/25 for the former and the latter.

In [1]:
import os
path = os.getcwd()
os.chdir(path + '/pytorch-generative')

In [2]:
import numpy as np

np.random.seed(256)

# Build the data set
TRSIZE = 50000
TESIZE = 10000
RVL = 10

p1 = 0.6
p2 = 0.4
prior = 0.75

training_d = np.zeros((TRSIZE, RVL))
for i in range(TRSIZE):
    if np.random.binomial(n=1, p=prior) == 1:
        training_d[i,:] = np.random.binomial(n=1, p=p1, size=RVL)
    else:
        training_d[i,:] = np.random.binomial(n=1, p=p2, size=RVL)

training_d = training_d.astype(np.single)
print(training_d.dtype)
        
testing_d = np.zeros((TESIZE, RVL))
for i in range(TESIZE):
    if np.random.binomial(n=1, p=prior) == 1:
        testing_d[i,:] = np.random.binomial(n=1, p=p1, size=RVL)
    else:
        testing_d[i,:] = np.random.binomial(n=1, p=p2, size=RVL)

testing_d = testing_d.astype(np.single)
print(testing_d.dtype)

print(training_d.shape)
print(testing_d.shape)

print(training_d)
print(testing_d)

float32
float32
(50000, 10)
(10000, 10)
[[1. 1. 1. ... 0. 1. 1.]
 [0. 0. 1. ... 0. 1. 1.]
 [0. 1. 0. ... 1. 1. 0.]
 ...
 [0. 0. 0. ... 1. 1. 0.]
 [1. 1. 1. ... 0. 1. 0.]
 [0. 1. 1. ... 1. 1. 0.]]
[[0. 1. 1. ... 0. 1. 1.]
 [0. 1. 1. ... 1. 0. 0.]
 [1. 0. 1. ... 0. 1. 0.]
 ...
 [1. 1. 0. ... 1. 1. 0.]
 [1. 1. 0. ... 0. 0. 1.]
 [0. 1. 1. ... 0. 0. 1.]]


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils

class bernDataset(Dataset):
    
    def __init__(self, variables, transform=None):
        self.variables = variables
        self.transform = transform
    
    def __len__(self):
        return len(self.variables)
    
    def __getitem__(self, idx):
        sample = self.variables[idx]
        
        if self.transform:
            sample = self.transform(sample)
        
        return sample

/Users/brucerushing/opt/anaconda3/envs/Python39/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: dlopen(/Users/brucerushing/opt/anaconda3/envs/Python39/lib/python3.9/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIxEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  Referenced from: <67CD63CE-57E0-341F-B3B8-78729B03D2B3> /Users/brucerushing/opt/anaconda3/envs/Python39/lib/python3.9/site-packages/torchvision/image.so
  Expected in:     <908CA0CD-2294-3FE1-BD48-F54A78458210> /Users/brucerushing/opt/anaconda3/envs/Python39/lib/python3.9/site-packages/torch/lib/libtorch_cpu.dylib
  warn(f"Failed to load image Python extension: {e}")


## Build the NADE Model

Now we build the neural autoregressive distribution estimator.  This code comes from Eugen Hotaj's pytorch_generative project.

In [4]:
from pytorch_generative.models.autoregressive.nade import NADE
from torch import optim
from torch.nn import functional as F
from pytorch_generative import trainer

log_dir = os.makedirs(os.path.join(os.getcwd(), 'checkpoint'))
epochs = 50

transformers = lambda x: torch.from_numpy(x).float()

tr_set = bernDataset(training_d, transform=transformers)
te_set = bernDataset(testing_d, transform=transformers)
tr_loader = torch.utils.data.DataLoader(tr_set, batch_size=100, shuffle=True)
te_loader = torch.utils.data.DataLoader(te_set, batch_size=100, shuffle=True)

model = NADE(input_dim=RVL, hidden_dim=100)
optimizer = optim.Adam(model.parameters())

def loss_fn(x, _, preds):
    batch_size = x.shape[0]
    x, preds = x.view((batch_size, -1)), preds.view((batch_size, -1))
    loss = F.binary_cross_entropy_with_logits(preds, x, reduction="none")
    return loss.sum(dim=1).mean()

model_trainer = trainer.Trainer(
                model=model,
                loss_fn=loss_fn,
                optimizer=optimizer,
                train_loader=tr_loader,
                eval_loader=te_loader,
                log_dir=log_dir,
                n_gpus=0,
                device_id=0)

model_trainer.interleaved_train_and_eval(epochs)

No checkpoint found in /var/folders/b9/gqhdnd8x7_s0c_sjpz884sg40000gn/T/tmpi8tv0fm5. Training from scratch.
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute '_c'
Failed to sample from the model: 'NADE' object has no attribute 

In [5]:
test_item = np.reshape(testing_d[0], (1, RVL))
test_permute = np.reshape(np.random.permutation(testing_d[0]), (1, RVL))
p_hat1 = model.forward(transformers(test_item))
p_hat2 = model.forward(transformers(test_permute))

In [6]:
print(p_hat1)
print(p_hat2)

tensor([[0.1922, 0.1120, 0.1653, 0.2402, 0.2243, 0.2179, 0.1908, 0.3097, 0.2294,
         0.2165]], grad_fn=<ViewBackward0>)
tensor([[0.1922, 0.2745, 0.2076, 0.2612, 0.1963, 0.2562, 0.2637, 0.1941, 0.3419,
         0.4540]], grad_fn=<ViewBackward0>)


In [7]:
print(torch.prod(p_hat1))
print(torch.prod(p_hat2))

tensor(1.2255e-07, grad_fn=<ProdBackward0>)
tensor(1.1425e-06, grad_fn=<ProdBackward0>)


In [8]:
test_item = np.reshape(testing_d[1], (1, RVL))
test_permute = np.reshape(np.random.permutation(testing_d[1]), (1, RVL))
p_hat1 = model.forward(transformers(test_item))
p_hat2 = model.forward(transformers(test_permute))

print(p_hat1)
print(p_hat2)
print(torch.prod(p_hat1))
print(torch.prod(p_hat2))

tensor([[0.1922, 0.1120, 0.1653, 0.2402, 0.2243, 0.3584, 0.2754, 0.4138, 0.3988,
         0.2708]], grad_fn=<ViewBackward0>)
tensor([[0.1922, 0.1120, 0.0835, 0.1140, 0.0122, 0.1896, 0.2334, 0.1150, 0.3717,
         0.3501]], grad_fn=<ViewBackward0>)
tensor(8.4548e-07, grad_fn=<ProdBackward0>)
tensor(1.6555e-09, grad_fn=<ProdBackward0>)


In [9]:
test_item = np.reshape(training_d[0], (1, RVL))
test_permute = np.reshape(np.random.permutation(training_d[0]), (1, RVL))
print(test_item)
print(test_permute)

p_hat1 = model.forward(transformers(test_item))
p_hat2 = model.forward(transformers(test_permute))

print(p_hat1)
print(p_hat2)
print(torch.prod(p_hat1))
print(torch.prod(p_hat2))

[[1. 1. 1. 1. 1. 1. 1. 0. 1. 1.]]
[[1. 1. 1. 1. 1. 1. 0. 1. 1. 1.]]
tensor([[0.1922, 0.2745, 0.2861, 0.3666, 0.3605, 0.3794, 0.4275, 0.3988, 0.3138,
         0.3184]], grad_fn=<ViewBackward0>)
tensor([[0.1922, 0.2745, 0.2861, 0.3666, 0.3605, 0.3794, 0.4275, 0.4264, 0.4648,
         0.2523]], grad_fn=<ViewBackward0>)
tensor(1.2896e-05, grad_fn=<ProdBackward0>)
tensor(1.6177e-05, grad_fn=<ProdBackward0>)


In [10]:
test_item = np.reshape(training_d[1], (1, RVL))
test_permute = np.reshape(np.random.permutation(training_d[1]), (1, RVL))
print(test_item)
print(test_permute)

p_hat1 = model.forward(transformers(test_item))
p_hat2 = model.forward(transformers(test_permute))

print(p_hat1)
print(p_hat2)
print(torch.prod(p_hat1))
print(torch.prod(p_hat2))

[[0. 0. 1. 0. 0. 0. 0. 0. 1. 1.]]
[[0. 0. 0. 1. 0. 1. 0. 0. 1. 0.]]
tensor([[0.1922, 0.1120, 0.0835, 0.1140, 0.0122, 0.0236, 0.0062, 0.0012, 0.0062,
         0.0055]], grad_fn=<ViewBackward0>)
tensor([[0.1922, 0.1120, 0.0835, 0.0131, 0.0341, 0.0184, 0.0303, 0.0311, 0.0072,
         0.0224]], grad_fn=<ViewBackward0>)
tensor(1.5493e-17, grad_fn=<ProdBackward0>)
tensor(2.2635e-15, grad_fn=<ProdBackward0>)


In [11]:
print(model._h_W)
print(model._h_b)

Parameter containing:
tensor([[ 1.7919e-01, -3.4653e-02, -2.0773e-01, -2.5353e-01, -5.3094e-02,
         -1.6097e-01, -2.0355e-02,  1.6214e-01, -7.4597e-02, -7.8310e-02,
         -1.3855e-01, -1.8185e-02, -2.6762e-01,  5.7893e-02, -2.5251e-01,
          3.4030e-02, -2.1735e-01,  6.2787e-03, -1.0815e-01, -4.1887e-02,
         -1.8287e-01, -1.5444e-01, -9.0460e-02, -2.7170e-01, -6.1774e-02,
         -6.3877e-02, -2.2572e-01, -3.4264e-03, -3.1625e-02, -8.1317e-02,
         -3.7595e-01, -1.5653e-01,  1.8106e-01,  1.3800e-01, -7.9002e-02,
          2.2447e-01, -1.9061e-01, -2.0872e-02, -6.2072e-02,  1.0076e-01,
         -8.5615e-02,  1.4529e-02, -5.3600e-03, -7.9920e-02, -3.9273e-01,
         -3.6777e-01, -1.8928e-02,  5.8250e-02, -1.9630e-01,  8.0285e-03,
         -8.7521e-02,  1.3548e-01, -7.4769e-02, -6.6324e-02, -4.8097e-02,
          1.3253e-03, -2.1397e-01, -1.6280e-01, -7.8113e-02,  1.3910e-01,
         -1.3548e-01, -1.3620e-01, -2.2902e-02, -1.1452e-01,  2.6027e-02,
          9.4768